# scenarios (chapter 8)

A monthly model of the vendor case: the ramp, the chance p that a found saving is banked and the whole cost of chapter 9. It draws Figures 8.1 and 8.2 and gives the payback months that Appendix B cannot (it works in years). All figures synthetic, USD.

In [ ]:
import pandas as pd
inp = pd.read_csv("../data/kestrel_inputs.csv").set_index("item")["value"]
inp

## The monthly cash model

Months 1 to 3 earn nothing, 4 to 6 half, 7 on in full. One-off costs (install, data preparation, year one people time) fall at month 0.

In [ ]:
recovered = inp["volume_hL_a_year"] * inp["cellar_loss_rate"] * inp["share_recovered_claim"]
running = 12 * (inp["licence_usd_a_month"] + inp["cloud_bill_usd_a_month"])
people_y1 = (13 * 4 + 39 * 1) * inp["loaded_rate_usd_an_hour"]
people_later = 52 * 1 * inp["loaded_rate_usd_an_hour"]
one_off = inp["install_usd"] + inp["data_prep_hours"] * inp["contractor_rate_usd_an_hour"] + people_y1

def earned_months(m):
    return 0 if m <= 3 else (0.5 * (m - 3) if m <= 6 else 1.5 + (m - 6))

def cash(p, value, months=36):
    return [recovered * value * p / 12 * earned_months(m) - (one_off + running / 12 * m + people_later / 12 * max(0, m - 12))
            for m in range(months + 1)]

def payback(series):
    return next((m for m, c in enumerate(series) if m > 0 and c >= 0), None)

## Conservative, base and stretch (8.3)

In [ ]:
later = running + people_later
rows = []
for name, p in [("Conservative", 0.25), ("Base", 0.45), ("Stretch", 0.70), ("Everything banked", 1.00)]:
    s = cash(p, inp["value_lost_hL_upper_bound_usd"])
    rows.append((name, p, round(recovered * 30 * p - later), round(recovered * 110 * p - later), round(s[36]), payback(s)))
pd.DataFrame(rows, columns=["Case", "p", "Steady net at USD 30", "Steady net at USD 110", "Cash at month 36, USD 110", "Payback month"])

## Figure 8.1: cumulative cash over 36 months

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
for name, p, v, style in [("Everything banked", 1.0, 110, "-"), ("Stretch", 0.70, 110, "-"),
                          ("Base", 0.45, 110, "--"), ("Conservative", 0.25, 110, ":"),
                          ("Replacement cost, everything banked", 1.0, 30, "-.")]:
    ax.plot(range(37), cash(p, v), style, label=name)
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("Month")
ax.set_ylabel("Cumulative cash, USD (synthetic)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print("Replacement cost, everything banked, month 36:", round(cash(1.0, 30)[36]))

## Break even as a share of the claim (8.4)

Steady and three year answers differ. Do not mix them.

In [ ]:
three_year_cost = one_off + 3 * running + 2 * people_later
upper = recovered * inp["value_lost_hL_upper_bound_usd"]
steady_be, three_be = later / upper, three_year_cost / (upper * 2.625)
print(f"Steady at USD 30: {later / (recovered * 30):.0%}; that is {later / 30:.0f} hL a year")
print(f"Steady at USD 110: {steady_be:.0%}; three years: {three_year_cost:,.0f} / {upper * 2.625:,.0f} = {three_be:.0%}")
s = 0.41
print(f"Share sold at peak s = {s}: {recovered * (30 + 80 * s) * 2.625:,.0f}")

ps = [0.2, 1.0]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(ps, [100 * p for p in ps], ":", color="grey", label="share banked = p")
ax.axhline(100 * three_be, label=f"three year case {three_be:.0%}")
ax.axhline(100 * steady_be, ls="--", label=f"steady case {steady_be:.0%}")
ax.scatter([0.25, 0.45, 0.70], [25, 45, 70], zorder=3)
ax.set_xlabel("p, the chance a found saving is banked")
ax.set_ylabel("Share of the claim that must land, percent")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Which input moves the answer most (8.5)

In [ ]:
def steady(vol=100_000, rate=0.024, share=0.20, value=110, p=0.45, licence=16_800):
    return vol * rate * share * value * p - (licence + 3_000 + people_later)

rows = [("p", steady(p=0.36), steady(p=0.54), steady(p=0.25), steady(p=0.70)),
        ("Value of a lost hL", steady(value=88), steady(value=132), steady(value=30), steady(value=110)),
        ("Share recovered", steady(share=0.16), steady(share=0.24), steady(share=0.10), steady(share=0.5 / 2.4)),
        ("Loss rate", steady(rate=0.0192), steady(rate=0.0288), steady(rate=0.020), steady(rate=0.028)),
        ("Volume", steady(vol=80_000), steady(vol=120_000), steady(vol=90_000), steady(vol=110_000)),
        ("Licence", steady(licence=13_440), steady(licence=20_160), steady(), steady())]
pd.DataFrame(rows, columns=["Input", "minus 20 percent", "plus 20 percent", "range low", "range high"]).round(0)